# Complete Workflow Validation Test

This notebook validates the entire 7-step document-link-factory workflow with custom documentation links.
Each step is executed sequentially and results are printed without function encapsulation.

In [15]:
import sys
import json
import os
from pathlib import Path
from datetime import datetime

# For Jupyter notebook compatibility
project_root = Path('/Users/gamer/Documents/daily-builder/doc2skill/project')
sys.path.insert(0, str(project_root / 'src'))
os.chdir(project_root)

from novae_skill.runtime import dispatch_action
from novae_skill.spec import (
    SkillConstraint,
    DEFAULT_DISCOVERY_HINTS,
    DEFAULT_EXCLUSION_HINTS,
)

print('✓ Libraries imported successfully')
print(f'  Project root: {project_root}')
print(f'  Working directory: {os.getcwd()}')
print(f'  Timestamp: {datetime.now()}')

✓ Libraries imported successfully
  Project root: /Users/gamer/Documents/daily-builder/doc2skill/project
  Working directory: /Users/gamer/Documents/daily-builder/doc2skill/project
  Timestamp: 2026-04-28 10:50:32.008034


## Step 1: Define Custom Links & Candidate Pages

Custom documentation links for workflow validation.

In [ ]:
import re
import requests
from urllib.parse import urljoin, urlparse

def extract_github_links(repo_url):
    """
    Extract tutorial and documentation links from a GitHub repository.
    
    Args:
        repo_url: GitHub repository URL (e.g., 'https://github.com/owner/repo')
    
    Returns:
        dict with extracted links and metadata
    """
    # Parse repo URL
    if 'github.com' not in repo_url:
        raise ValueError("Invalid GitHub repo URL")
    
    # Normalize repo URL
    repo_url = repo_url.rstrip('/')
    parts = repo_url.replace('https://', '').replace('http://', '').split('/')
    owner, repo = parts[1], parts[2]
    
    links = []
    api_base = f'https://api.github.com/repos/{owner}/{repo}'
    raw_base = f'https://raw.githubusercontent.com/{owner}/{repo}/main'
    
    print(f"📦 Extracting from GitHub: {owner}/{repo}")
    print(f"   API Base: {api_base}")
    
    # 1. Get README content
    readme_files = ['README.md', 'readme.md', 'README.rst', 'readme.rst']
    for readme in readme_files:
        readme_url = f'{raw_base}/{readme}'
        try:
            resp = requests.get(readme_url, timeout=5)
            if resp.status_code == 200:
                readme_content = resp.text
                
                # Extract markdown links from README
                md_links = re.findall(r'\[([^\]]+)\]\(([^)]+)\)', readme_content)
                print(f"\n✓ Found README.md with {len(md_links)} links")
                
                for link_title, link_url in md_links:
                    # Filter for tutorial-like links
                    if any(keyword in link_title.lower() or keyword in link_url.lower() 
                           for keyword in ['tutorial', 'guide', 'example', 'quickstart', 'demo', 'doc']):
                        # Convert relative to absolute URLs
                        if link_url.startswith('#'):
                            abs_url = f"{repo_url}#" + link_url[1:]
                        elif link_url.startswith('/'):
                            abs_url = f"https://github.com{link_url}"
                        elif not link_url.startswith('http'):
                            abs_url = urljoin(f"{repo_url}/blob/main/", link_url)
                        else:
                            abs_url = link_url
                        
                        links.append({
                            'title': link_title,
                            'url': abs_url,
                            'kind': 'tutorial' if 'tutorial' in link_title.lower() else 'guide',
                            'source': 'github_readme'
                        })
                break
        except Exception as e:
            print(f"  Note: Could not fetch {readme}: {e}")
            continue
    
    # 2. Get releases and check for documentation links
    try:
        releases_url = f'{api_base}/releases?per_page=3'
        resp = requests.get(releases_url, timeout=5)
        if resp.status_code == 200:
            releases = resp.json()
            for release in releases:
                # Look for documentation links in release body
                body = release.get('body', '')
                md_links = re.findall(r'\[([^\]]+)\]\(([^)]+)\)', body)
                for link_title, link_url in md_links:
                    if any(keyword in link_title.lower() or keyword in link_url.lower() 
                           for keyword in ['doc', 'tutorial', 'guide', 'example']):
                        links.append({
                            'title': f"{link_title} (Release)",
                            'url': link_url,
                            'kind': 'guide',
                            'source': 'github_release'
                        })
    except Exception as e:
        print(f"  Note: Could not fetch releases: {e}")
    
    # 3. Check common documentation directories
    doc_paths = ['docs/', 'doc/', 'documentation/', 'examples/', 'tutorials/']
    repo_web_url = f"https://github.com/{owner}/{repo}/tree/main"
    
    for doc_dir in doc_paths:
        doc_url = f"{repo_web_url}/{doc_dir}".rstrip('/')
        links.append({
            'title': f"{doc_dir.rstrip('/')} directory",
            'url': doc_url,
            'kind': 'documentation',
            'source': 'github_structure',
            'priority': 'high'
        })
    
    # 4. Add main entry page
    entry = {
        'title': f"{repo} Documentation",
        'url': repo_url,
        'kind': 'documentation',
        'source': 'github_repo',
        'priority': 'high'
    }
    links.insert(0, entry)
    
    return {
        'repo_url': repo_url,
        'owner': owner,
        'repo_name': repo,
        'links': links,
        'total_extracted': len(links)
    }

# Example: Extract from a real GitHub repo
github_repo_url = 'https://github.com/uhlerlab/spatialfusion'  # Example: Python official repo

try:
    github_result = extract_github_links(github_repo_url)
    print(f"\n✓ Extracted {github_result['total_extracted']} total links")
    print(f"\nDiscovered Links:")
    for link in github_result['links'][:8]:  # Show first 8
        print(f"\n  • {link['title']}")
        print(f"    URL: {link['url']}")
        print(f"    Kind: {link.get('kind', 'unknown')} | Source: {link.get('source', 'unknown')}")
except Exception as e:
    print(f"✗ Error extracting GitHub links: {e}")

✗ Error extracting GitHub links: Invalid GitHub repo URL


## Option: Load From GitHub Repository

Automatically extract tutorial links and documentation from a GitHub repository.

In [3]:
# Custom documentation entry URL and candidate pages
entry_url = 'https://github.com/uhlerlab/spatialfusion'

# candidate_pages = [
#     {
#         'title': 'Python Basics Tutorial',
#         'url': 'https://docs.example.com/python-guide/basics',
#         'kind': 'tutorial',
#         'priority': 'high',
#         'evidence': ['sidebar', 'code_examples'],
#         'reason': 'Official getting started tutorial with basic concepts'
#     },
#     {
#         'title': 'Advanced Features Guide',
#         'url': 'https://docs.example.com/python-guide/advanced',
#         'kind': 'guide',
#         'priority': 'high',
#         'evidence': ['multiple_sections', 'code_samples'],
#         'reason': 'Comprehensive guide for advanced users'
#     },
#     {
#         'title': 'API Reference',
#         'url': 'https://docs.example.com/python-guide/api-reference',
#         'kind': 'reference',
#         'priority': 'medium',
#         'evidence': ['parameter_tables'],
#         'reason': 'API documentation with parameter details'
#     },
#     {
#         'title': 'Quickstart Example',
#         'url': 'https://docs.example.com/python-guide/quickstart',
#         'kind': 'quickstart',
#         'priority': 'high',
#         'evidence': ['code_block', 'instructions'],
#         'reason': '5-minute quick start guide'
#     },
#     {
#         'title': 'Release Notes v2.0',
#         'url': 'https://docs.example.com/python-guide/changelog',
#         'kind': 'changelog',
#         'priority': 'low',
#         'evidence': ['version_history'],
#         'reason': 'Release notes and version history'
#     }
# ]

print('=' * 60)
print('CUSTOM DOCUMENTATION LINKS')
print('=' * 60)
print(f'\nEntry URL: {entry_url}')
# print(f'\nCandidate Pages ({len(candidate_pages)} total):')
# for i, page in enumerate(candidate_pages, 1):
#     print(f'\n  [{i}] {page["title"]}')
#     print(f'      URL: {page["url"]}')
#     print(f'      Kind: {page["kind"]} | Priority: {page["priority"]}')
#     print(f'      Evidence: {", ".join(page["evidence"])}')

CUSTOM DOCUMENTATION LINKS

Entry URL: https://github.com/uhlerlab/spatialfusion


In [4]:
# Choose link source: 'custom' or 'github'
LINK_SOURCE = 'github'  # Change to 'github' to use GitHub extraction

if LINK_SOURCE == 'github':
    # Use GitHub extracted links
    github_links = github_result['links']
    
    # Transform to candidate_pages format
    candidate_pages = []
    for i, link in enumerate(github_links, 1):
        candidate_pages.append({
            'title': link['title'],
            'url': link['url'],
            'kind': link.get('kind', 'documentation'),
            'priority': link.get('priority', 'medium'),
            'evidence': [link.get('source', 'github')],
            'reason': f"Extracted from {link.get('source', 'GitHub repository')}"
        })
    
    print('=' * 60)
    print('USING GITHUB-EXTRACTED LINKS')
    print('=' * 60)
    print(f'Repository: {github_result["repo_url"]}')
    print(f'Total links: {len(candidate_pages)}')
else:
    print('=' * 60)
    print('USING CUSTOM LINKS')
    print('=' * 60)
    print(f'Total links: {len(candidate_pages)}')

print(f'\nEntry URL: {entry_url if LINK_SOURCE == "custom" else github_result["repo_url"]}')

USING GITHUB-EXTRACTED LINKS
Repository: https://github.com/uhlerlab/spatialfusion
Total links: 8

Entry URL: https://github.com/uhlerlab/spatialfusion


## Step 1b: Choose Link Source

Select whether to use custom links or GitHub-extracted links for the workflow.

## Step 2: Validate Link Validity

Check if URLs are well-formed and accessible (basic validation).

In [5]:
from urllib.parse import urlparse

print('\n' + '=' * 60)
print('LINK VALIDITY CHECK')
print('=' * 60)

all_urls_valid = True
for page in candidate_pages:
    url = page['url']
    try:
        parsed = urlparse(url)
        has_scheme = bool(parsed.scheme)
        has_netloc = bool(parsed.netloc)
        
        if has_scheme and has_netloc:
            status = '✓ Valid'
        else:
            status = '✗ Invalid'
            all_urls_valid = False
            
        print(f'\n{status}: {url}')
        print(f'  Scheme: {parsed.scheme}, Domain: {parsed.netloc}, Path: {parsed.path}')
    except Exception as e:
        print(f'\n✗ Error parsing {url}: {e}')
        all_urls_valid = False

print(f'\n{"✓" if all_urls_valid else "✗"} Overall link validity: {"All links are well-formed" if all_urls_valid else "Some links have issues"}')


LINK VALIDITY CHECK

✓ Valid: https://github.com/uhlerlab/spatialfusion
  Scheme: https, Domain: github.com, Path: /uhlerlab/spatialfusion

✓ Valid: https://img.shields.io/badge/docs-online-blue.svg
  Scheme: https, Domain: img.shields.io, Path: /badge/docs-online-blue.svg

✓ Valid: https://uhlerlab.github.io/spatialfusion/unimodal-embeddings/
  Scheme: https, Domain: uhlerlab.github.io, Path: /spatialfusion/unimodal-embeddings/

✓ Valid: https://github.com/uhlerlab/spatialfusion/tree/main/docs
  Scheme: https, Domain: github.com, Path: /uhlerlab/spatialfusion/tree/main/docs

✓ Valid: https://github.com/uhlerlab/spatialfusion/tree/main/doc
  Scheme: https, Domain: github.com, Path: /uhlerlab/spatialfusion/tree/main/doc

✓ Valid: https://github.com/uhlerlab/spatialfusion/tree/main/documentation
  Scheme: https, Domain: github.com, Path: /uhlerlab/spatialfusion/tree/main/documentation

✓ Valid: https://github.com/uhlerlab/spatialfusion/tree/main/examples
  Scheme: https, Domain: github.

## Step 3-7: Execute All Workflow Actions

Run through all 7 actions in sequence.

In [11]:
# ACTION 1: Discover
print('\n' + '=' * 60)
print('ACTION 1: DISCOVER_DOCUMENT_PAGES')
print('=' * 60)
discover_result = dispatch_action('discover_document_pages', {
    'entry_url': entry_url,
    'candidate_links': candidate_pages,
    'navigation_hints': DEFAULT_DISCOVERY_HINTS,
    'exclusion_hints': DEFAULT_EXCLUSION_HINTS,
})
print(f'\nPages Discovered: {discover_result["page_count"]}')

# ACTION 2: Classify
print('\n' + '=' * 60)
print('ACTION 2: CLASSIFY_DOCUMENT_PAGES')
print('=' * 60)
classify_result = dispatch_action('classify_document_pages', {
    'pages': discover_result['discovered_pages'],
    'keep_kinds': ['tutorial', 'guide', 'example', 'quickstart', 'walkthrough', 'how-to', 'documentation'],
    'exclude_kinds': ['marketing', 'blog', 'changelog', 'release notes', 'reference'],
})
print(f'\nKept: {classify_result["kept_count"]} | Excluded: {classify_result["excluded_count"]}')

print(f'\n✓ INCLUDED Pages:')
for page in classify_result['included_pages']:
    print(f'  - {page["title"]} [{page["kind"]}]')

print(f'\n✗ EXCLUDED Pages:')
for page in classify_result['excluded_pages']:
    print(f'  - {page["title"]} [{page["kind"]}]')


ACTION 1: DISCOVER_DOCUMENT_PAGES

Pages Discovered: 8

ACTION 2: CLASSIFY_DOCUMENT_PAGES

Kept: 8 | Excluded: 0

✓ INCLUDED Pages:
  - spatialfusion Documentation [documentation]
  - ![Docs [guide]
  - documentation website [guide]
  - docs directory [documentation]
  - doc directory [documentation]
  - documentation directory [documentation]
  - examples directory [documentation]
  - tutorials directory [documentation]

✗ EXCLUDED Pages:


In [12]:
# ACTION 3: Extract Capabilities
print('\n' + '=' * 60)
print('ACTION 3: EXTRACT_DOCUMENT_CAPABILITIES')
print('=' * 60)

page_evidence = [
    {
        'name': 'python_installation_setup',
        'purpose': 'Guide users through installing and setting up Python',
        'inputs': ['OS_type', 'python_version_preference'],
        'outputs': ['installation_path', 'verification_status'],
        'dependencies': ['system_package_manager', 'official_installer'],
        'risk_points': ['PATH environment variable must be updated'],
        'evidence_pages': ['https://docs.example.com/python-guide/basics'],
        'notes': ['Covers Windows, macOS, Linux']
    },
    {
        'name': 'decorator_implementation',
        'purpose': 'Teach how to create and use decorators',
        'inputs': ['function_to_decorate', 'decorator_logic'],
        'outputs': ['decorated_function', 'execution_trace'],
        'dependencies': ['function_objects', 'closure_concept'],
        'risk_points': ['Must preserve function metadata with functools.wraps'],
        'evidence_pages': ['https://docs.example.com/python-guide/advanced'],
        'notes': ['Includes examples of decorator with/without arguments']
    }
]

extract_result = dispatch_action('extract_document_capabilities', {
    'pages': classify_result['included_pages'],
    'page_evidence': page_evidence,
    'target_language': 'python',
    'constraints': {'allow_third_party_libs': True}
})

print(f'\nCapabilities Extracted: {len(extract_result["capabilities"])}')
for cap in extract_result['capabilities']:
    print(f'\n  • {cap["name"]}')
    print(f"    Purpose: {cap['purpose']}")
    print(f"    Inputs: {', '.join(cap['inputs'][:2])}{'...' if len(cap['inputs']) > 2 else ''}")

# ACTION 4: Normalize
print('\n' + '=' * 60)
print('ACTION 4: NORMALIZE_CAPABILITIES')
print('=' * 60)

normalize_result = dispatch_action('normalize_capabilities', {
    'capabilities': extract_result['capabilities'],
    'merge_strategy': 'collapse_duplicates',
    'dedupe_keys': ['name']
})

print(f'\nNormalized Capabilities: {len(normalize_result["normalized_capabilities"])}')
for cap in normalize_result['normalized_capabilities']:
    print(f'  • {cap["name"]}: {len(set(cap["inputs"]))} unique inputs, {len(set(cap["outputs"]))} unique outputs')


ACTION 3: EXTRACT_DOCUMENT_CAPABILITIES

Capabilities Extracted: 2

  • python_installation_setup
    Purpose: Guide users through installing and setting up Python
    Inputs: OS_type, python_version_preference

  • decorator_implementation
    Purpose: Teach how to create and use decorators
    Inputs: function_to_decorate, decorator_logic

ACTION 4: NORMALIZE_CAPABILITIES

Normalized Capabilities: 2
  • python_installation_setup: 2 unique inputs, 2 unique outputs
  • decorator_implementation: 2 unique inputs, 2 unique outputs


In [13]:
# ACTION 5: Design Package
print('\n' + '=' * 60)
print('ACTION 5: DESIGN_SKILL_PACKAGE')
print('=' * 60)

design_result = dispatch_action('design_skill_package', {
    'entry_url': entry_url,
    'project_name': 'spatialfusion',
    'skill_name': 'spatialfusion',
    'constraints': {
        'target_language': 'python',
        'target_skill_format': 'openapi',
        'allow_third_party_libs': True,
        'enable_local_cache': False,
        'enable_vector_index': False,
        'output_dir': './project/output/spatialfusion_skill'
    },
    'normalized_pages': classify_result['included_pages'],
    'capability_model': normalize_result['normalized_capabilities']
})

package_plan = design_result['package_plan']
print(f'\nPackage Plan: {package_plan["project_name"]}')
print(f'  Skill: {package_plan["skill_name"]}')
print(f'  Pages: {len(package_plan["included_pages"])} included')
print(f'  Capabilities: {len(package_plan["capability_model"])}')
print(f'  Files: {len(package_plan["file_tree"])} files')

# ACTION 6: Generate Files
print('\n' + '=' * 60)
print('ACTION 6: GENERATE_SKILL_FILES')
print('=' * 60)

generate_result = dispatch_action('generate_skill_files', {
    'package_plan': package_plan
})

files = generate_result['files']
print(f'\nGenerated Files: {len(files)}')
print(f'Total Size: {sum(len(c) for c in files.values())} bytes')
for file_path in sorted(files.keys())[:5]:
    print(f'  • {file_path}')

# ACTION 7: Validate
print('\n' + '=' * 60)
print('ACTION 7: VALIDATE_SKILL_PACKAGE')
print('=' * 60)

validate_result = dispatch_action('validate_skill_package', {
    'package_plan': package_plan,
    'files': files
})

print(f'\nValidation Result: {"✓ Valid" if validate_result["valid"] else "✗ Invalid"}')
print(f'Missing Files: {len(validate_result["missing_files"])}')
print(f'Warnings: {len(validate_result["warnings"])}')

print('\n✓ COMPLETE WORKFLOW EXECUTION SUCCESSFUL')


ACTION 5: DESIGN_SKILL_PACKAGE

Package Plan: spatialfusion
  Skill: spatialfusion
  Pages: 8 included
  Capabilities: 2
  Files: 14 files

ACTION 6: GENERATE_SKILL_FILES

Generated Files: 14
Total Size: 3270 bytes
  • project/.env.example
  • project/README.md
  • project/SKILL.md
  • project/examples/discover_and_generate.py
  • project/examples/package_validation.py

ACTION 7: VALIDATE_SKILL_PACKAGE

Validation Result: ✓ Valid
Missing Files: 0
Warnings: 0

✓ COMPLETE WORKFLOW EXECUTION SUCCESSFUL


In [14]:
# BONUS: Save Generated Files to Disk
print('\n' + '=' * 60)
print('SAVING GENERATED FILES TO DISK')
print('=' * 60)

output_dir = Path('./project/output/spatialfusion_skill')
output_dir.mkdir(parents=True, exist_ok=True)

print(f'\nOutput Directory: {output_dir.absolute()}')
print(f'\nSaving {len(files)} files...\n')

saved_count = 0
for file_path, file_content in files.items():
    full_path = output_dir / file_path
    full_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Write file
    if isinstance(file_content, str):
        with open(full_path, 'w') as f:
            f.write(file_content)
    else:
        with open(full_path, 'wb') as f:
            f.write(file_content if isinstance(file_content, bytes) else str(file_content).encode())
    
    saved_count += 1
    print(f'  ✓ {file_path} ({len(file_content) if isinstance(file_content, (str, bytes)) else 0} bytes)')

print(f'\n✓ Successfully saved {saved_count} files to:')
print(f'  {output_dir.absolute()}')

# List saved files
print(f'\nGenerated Files Structure:')
for root, dirs, filenames in os.walk(output_dir):
    level = root.replace(str(output_dir), '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = ' ' * 2 * (level + 1)
    for filename in filenames:
        print(f'{sub_indent}{filename}')


SAVING GENERATED FILES TO DISK

Output Directory: /Users/gamer/Documents/daily-builder/doc2skill/project/project/output/spatialfusion_skill

Saving 14 files...

  ✓ project/SKILL.md (476 bytes)
  ✓ project/README.md (265 bytes)
  ✓ project/pyproject.toml (268 bytes)
  ✓ project/.env.example (132 bytes)
  ✓ project/schema/openapi.json (58 bytes)
  ✓ project/src/docskill_factory/__init__.py (37 bytes)
  ✓ project/src/docskill_factory/spec.py (36 bytes)
  ✓ project/src/docskill_factory/runtime.py (36 bytes)
  ✓ project/src/docskill_factory/renderer.py (36 bytes)
  ✓ project/tests/test_spec.py (491 bytes)
  ✓ project/tests/test_runtime.py (581 bytes)
  ✓ project/tests/notebook_tests.ipynb (75 bytes)
  ✓ project/examples/discover_and_generate.py (385 bytes)
  ✓ project/examples/package_validation.py (394 bytes)

✓ Successfully saved 14 files to:
  /Users/gamer/Documents/daily-builder/doc2skill/project/project/output/spatialfusion_skill

Generated Files Structure:
spatialfusion_skill/
  pro